# SpecViz Full Pipeline - Debugging Notebook
**Purpose**: Debug the SpecViz Streamlit app by running the exact same pipeline without GUI components.
## 1. Edit the following Parameters:
- `project_name` → this determines under which folder you will find your results
- `input_folders` → list of the folders from which the script will first load all of the idXML and then the mzML files
- `peptides` → list of all the peptides to filter the idxml (!Need to be in "quotes")
- `reference_mzml_files` → replaces reference file selector
- `max_plots`, `save_plots` → replaces plot configuration UI (maximum number of plots & whether to save them or not)
- `ion_types` → list of ion types to choose from (a,b,c,x,y,z)
- `fragment_mass_tolerance` → leave as default unless the peaks are not being anotated (if so increase)

## 2. Save & Run 
- To save the notbook either press Ctrl+S or in VSCode click under File>Save (Tipp: I recommend to save an editted copy of this notebook)
- Once saved, you can start the pipeline by clicking on the top button Run All or enter the following command in the VSCode searchbar: `>Notebook: Run All` 


In [1]:
# ========================================================================
# PIPELINE PARAMETERS - Configure everything here
# ========================================================================

# === PROJECT NAME ===
project_name = "Anna_ProCa_classI_mini2"

# === INPUT FOLDERS ===
input_folders = [
    #"/mnt/volume/nacho/data/ProCa_Anna/mzml_classI/",
    #"/mnt/volume/nacho/data/ProCa_Anna/idxml_classI/",
    "/mnt/volume/nacho/data/ProCa_Anna/small_dataset/",
]

# === TARGET PEPTIDES (unmodified sequences) ===
peptides = [
    "KPFDQTTISL","ATELTHEDY", "AYIKNGQLF", "ATYSVGMQK"


    # "AEYHAKATEHLSTL", "NLSYNRITSPQVHR", "TIGEGQQHHLGGAKQAGDV", "TPEAGYFAVAVVKKSA", "YSLSSVVTVPSSSLGTQT", "FVTNPDGSPAYRVPVAVQGED",
    

    #"RTRWDSELQREGVSHYRLF","YSSDTRTEFNLNQYYQRDE", "LFLALYDSVILLKHVVLLL", "TELRRVLQGLEIELQSQLS", "LSPIDKNWPSRPYLFLDST", "AQRLDPVYFVAPAKFLGNQ",
    #"SMLIFILPSAFYIKLVKKE", "DLTVTSAHDQSLVLKQNLT", "FSRLYEGEADIMISFAVRE","IELLRKEEPGAFVIRDSSS","DLRNKILTATVDNANVL","DLRNQLLNYRSAISNHG", "GDLELRPSTFLIIDPL"

    #"RTEFNLNQY", "DSELQREGVSHY", "GTDRLLIEY", "DTDTNAQITY", "ASDDFDQASY", "ALLGSAFQL", "ALYDSVILL", "VLQGLEIEL",
    #"GMFGKETYV", "QLLEFNGINL", "ALMDEINFMKM", "KLAEISLGV", "VLVPYEPPQV", "QILDGVHYL", "VYFVAPAKF", "KYIHFTPNF",
    #"NWPSRPYLF", "KYLEKYYNL", "AYNAVSFTW", "KYGVHEAIF", "LTRAGILTF", "SQAFLVSIY", "PVFPTELAF", "ALFAVFQTTF",
    #"TAKSATWTY", "SVYRTVIEY", "VTSSVSSGY", "YVAHERLSY", "LPSAFYIKL", "LPGTAPKLLIY", "LPWAEFESL", "VPSEIDAAF",
    #"SAHDQSLVL", "GEADIMISF", "HEFSSPSHLL", "SETRVLVDI", "KEEPGAFVI", "SEGSPLVVL", "DLRNQLLNYRSAISNHG", 
    #"KDTLKKMQKFFGLPQ", "VPRKTWWASKSPDNKPVW", "FPGPNIGGDTHFDSAEPWTVRNEDL", "KDTSIAYFKNKELEQG",
    #"FKRGKLLSYVDAEGNPVG", "GDLELRPSTFLIIDPL", "IQDFIEAEDDLSSFR", "DLRNKILTATVDNANVL",
    #"IRQFTSSSSIKGSSG", "ERPKTSVNLISSLWPTLPSG", "FEQYINNLRRQLDSIVG",
]

# === REFERENCE MZML FILES (for One-vs-All comparison) ===
reference_mzml_files = [
    "/mnt/volume/nacho/data/ProCa_Anna/mzml_classI/251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML",
]

# === PLOT CONFIGURATION ===
max_plots = None  # Maximum number of plots to generate (None = all)
ion_types = ['b', 'y']  # Ion types to colour
save_plots = True  # Save plots to disk?
fragment_mass_tolerance = 0.02  # Fragment m/z tolerance in Da

## Setup and Imports

In [2]:
from pathlib import Path
import sys
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
import warnings
warnings.filterwarnings('ignore')

# Add SpecViz paths for imports
cwd = Path.cwd().resolve()
# Try to find specviz directory by looking for "utils" and "core" folders
specviz_dir = None
for parent in [cwd] + list(cwd.parents):
    if (parent / "utils").exists() and (parent / "core").exists():
        specviz_dir = parent
        break

if specviz_dir is None:
    raise RuntimeError("Could not locate specviz directory. Run from repo root or specviz/examples.")

sys.path.insert(0, str(specviz_dir / "utils"))
sys.path.insert(0, str(specviz_dir / "core"))
sys.path.insert(0, str(specviz_dir / "workspaces"))

print(f"✅ SpecViz directory: {specviz_dir}")

✅ SpecViz directory: /mnt/volume/nacho/specviz/spectrum_visualizer


In [3]:
# Import core modules
from shared_imports import pd, oms, Peptidoform

# CRITICAL: Mock Streamlit cache decorators BEFORE importing any modules that use them
# This prevents "No runtime found" warnings when running in Jupyter notebooks
import sys
from unittest.mock import MagicMock

# Create mock Streamlit module if it doesn't exist or override it
if 'streamlit' not in sys.modules:
    sys.modules['streamlit'] = MagicMock()

# Create no-op cache decorators that just return the function unchanged
def noop_decorator(func=None, **kwargs):
    """No-op decorator for Streamlit cache functions in notebook context."""
    if func is None:
        # Called with arguments: @cache_data(ttl=3600)
        return lambda f: f
    else:
        # Called without arguments: @cache_data
        return func

# Override Streamlit cache decorators with no-ops
sys.modules['streamlit'].cache_data = noop_decorator
sys.modules['streamlit'].cache_resource = noop_decorator

# Now import spectrum_analysis - cache decorators will be no-ops
import spectrum_analysis

# Reload the module to pick up the bug fix for bytes decoding
import importlib
importlib.reload(spectrum_analysis)

# IMPORTANT: Patch the cached loader to avoid pickling pyOpenMS objects in notebooks
# The app uses @st.cache_data which pickles return values, but pyOpenMS objects can't be pickled
# This replacement allows the notebook to work without Streamlit's caching
def _load_idxml_peptides_no_cache(idxml_path):
    protein_ids = []
    peptide_ids = []
    oms.IdXMLFile().load(idxml_path, protein_ids, peptide_ids)
    return protein_ids, peptide_ids

spectrum_analysis._load_idxml_peptides = _load_idxml_peptides_no_cache

# Import the EXACT same pipeline functions used by the app
from spectrum_analysis import (
    filter_idxml_by_selected_peptides,  # PHASE 2: Filter idXMLs by peptide list
    load_only_needed_spectra,           # PHASE 3: Load only needed spectra (memory-efficient)
)

# Import the EXACT same plotting functions used by the app
from spectrum_engine import (
    create_matplotlib_mirror_plot,      # Create mirror plot figure
    export_matplotlib_mirror_plot,      # Export plot to file
    convert_spectrum_to_df,             # Convert spectrum to DataFrame
    generate_theoretical_spectrum,      # Generate theoretical spectrum
    align_spectra,                      # Align experimental and theoretical spectra
    annotate_peaks,                     # Annotate matched peaks
    extract_annotations,                # Extract annotations for plotting
    create_peptide_overview_plot,       # Create summary barplot
    export_overview_plot,               # Export summary plot
    EXP_COLORS,                         # Color scheme for experimental spectra
    PRED_COLORS,                        # Color scheme for predicted spectra
)

from correlation import spectrum_correlation, new_spectrum_correlation

print("✅ Modules loaded successfully (Streamlit cache decorators disabled for notebook)")


✅ Modules loaded successfully (Streamlit cache decorators disabled for notebook)


In [4]:
# === OUTPUT DIRECTORIES ===
plot_output_dir = specviz_dir / "plots" / project_name
analysis_output_dir = specviz_dir / "analysis" / project_name

# Create output directories
plot_output_dir.mkdir(parents=True, exist_ok=True)
analysis_output_dir.mkdir(parents=True, exist_ok=True)


## Helper Functions

In [5]:
# ===================================================================
# HELPER FUNCTIONS - Minimal wrappers around app functions
# These functions exist in the notebook to:
# 1. Provide convenient interfaces for notebook usage
# 2. Bridge between notebook parameters and app function signatures
# 3. Add debug output where helpful
# ALL core logic is delegated to spectrum_analysis.py functions
# ===================================================================

def collect_mzml_and_idxml(input_folders):
    """Recursively collect all mzML and idXML files from input folders.
    
    This is a notebook-specific helper that replaces the file browser UI.
    In the app, users browse/upload files instead.
    
    Args:
        input_folders: List of folder paths to search
    
    Returns:
        (mzml_paths, idxml_paths): Lists of file paths
    """
    mzml_paths = []
    idxml_paths = []
    
    for folder in input_folders:
        folder_path = Path(folder)
        if not folder_path.exists():
            raise FileNotFoundError(f"Folder not found: {folder}")
        
        for path in folder_path.rglob("*"):
            if not path.is_file():
                continue
            
            suffix = path.suffix.lower()
            if suffix == ".mzml":
                mzml_paths.append(str(path))
            elif suffix in {".idxml", ".mzid", ".mzidentml"}:
                idxml_paths.append(str(path))
    
    return sorted(set(mzml_paths)), sorted(set(idxml_paths))


def resolve_reference_idxml(reference_idxml, idxml_paths):
    """Resolve reference idXML path from string (can be full path or filename).
    
    This replaces the app's reference file selector UI.
    
    Args:
        reference_idxml: Full path or filename of reference idXML
        idxml_paths: List of all available idXML paths
    
    Returns:
        Resolved full path to reference idXML
    
    Raises:
        ValueError: If reference not found or ambiguous
    """
    reference_path = Path(reference_idxml)
    
    # If it's a full path and exists
    if reference_path.exists():
        resolved = str(reference_path)
        if resolved not in idxml_paths:
            raise ValueError(f"Reference idXML is not in input folders: {reference_idxml}")
        return resolved
    
    # Try to match by filename
    matches = [p for p in idxml_paths if Path(p).name == reference_idxml]
    if not matches:
        raise ValueError(f"Reference idXML not found by name: {reference_idxml}")
    if len(matches) > 1:
        raise ValueError(f"Reference idXML name is ambiguous (found {len(matches)} matches): {reference_idxml}")
    
    return matches[0]


def build_spectrum_table_from_filtered(idxml_to_mzml_spectra):
    """Build spectrum table from filtered idXML data.
    
    This creates a simple table view of what was filtered.
    In the app, this information is shown in the UI.
    
    Args:
        idxml_to_mzml_spectra: Dict from filter_idxml_by_selected_peptides
    
    Returns:
        DataFrame with columns: spectrum_id, peptidoform, peptide_sequence, mzml_file, idxml_source
    """
    rows = []
    
    for idxml_name, mzml_map in idxml_to_mzml_spectra.items():
        for mzml_file, spec_id_peptide_pairs in mzml_map.items():
            for item in spec_id_peptide_pairs:
                spec_id = item[0]
                peptidoform = item[1]
                try:
                    unmodified = Peptidoform(peptidoform).sequence
                except Exception:
                    unmodified = ""
                
                rows.append({
                    "spectrum_id": spec_id,
                    "peptidoform": peptidoform,
                    "peptide_sequence": unmodified,
                    "mzml_file": mzml_file,
                    "idxml_source": idxml_name,
                })
    
    return pd.DataFrame(rows)


def load_indexed_experiments_for_plotting(mzml_paths):
    """Load mzML files with indexed access (memory-efficient).
    
    Uses the same OnDiscMSExperiment approach as the app.
    
    Args:
        mzml_paths: List of mzML file paths
    
    Returns:
        Dict of {mzml_filename: OnDiscMSExperiment}
    """
    experiments = {}
    
    for mzml_path in mzml_paths:
        try:
            exp = oms.OnDiscMSExperiment()
            exp.openFile(str(mzml_path))
            experiments[Path(mzml_path).name] = exp
            print(f"  ✅ Opened {Path(mzml_path).name} ({exp.getNrSpectra()} spectra)")
        except Exception as e:
            print(f"  ❌ Failed to open {Path(mzml_path).name}: {e}")
    
    return experiments


def build_spectrum_lookups(experiments):
    """Build spectrum lookups for efficient spectrum retrieval.
    
    Uses the same lookup strategy as the app.
    
    Args:
        experiments: Dict of {mzml_name: experiment_object}
    
    Returns:
        Dict of {mzml_stem: lookup_object}
    """
    lookups = {}
    
    for mzml_name, exp in experiments.items():
        try:
            # Try standard SpectrumLookup first
            first_native_id = exp.getSpectrum(0).getNativeID() if exp.getNrSpectra() > 0 else ""
            
            # Detect pattern
            if "scan=" in first_native_id:
                pattern = "scan=(?<SCAN>\\d+)"
            elif "index=" in first_native_id:
                pattern = "index=(?<SCAN>\\d+)"
            elif "spectrum=" in first_native_id:
                pattern = "spectrum=(?<SCAN>\\d+)"
            else:
                pattern = "(?<SCAN>\\d+)"
            
            lookup = oms.SpectrumLookup()
            lookup.readSpectra(exp, pattern)
            lookups[Path(mzml_name).stem] = lookup
            
        except Exception as exc:
            # Fallback for OnDiscMSExperiment: build native_id -> index map
            try:
                native_id_map = {}
                for idx in range(exp.getNrSpectra()):
                    native_id = exp.getSpectrum(idx).getNativeID()
                    if native_id:
                        native_id_map[native_id] = idx
                lookups[Path(mzml_name).stem] = native_id_map
            except Exception as fallback_exc:
                print(f"  ⚠️ Failed to build lookup for {mzml_name}: {exc}")
    
    return lookups


def load_spectrum_from_experiment(experiments, lookups, mzml_file, spectrum_id):
    """Load a specific spectrum from experiments.
    
    Uses the same spectrum loading logic as the app.
    
    Args:
        experiments: Dict of {mzml_filename: experiment}
        lookups: Dict of {mzml_stem: lookup}
        mzml_file: mzML filename
        spectrum_id: Native spectrum ID
    
    Returns:
        MSSpectrum object or None
    """
    exp = experiments.get(mzml_file)
    lookup = lookups.get(Path(mzml_file).stem)
    
    if exp is None or lookup is None:
        return None
    
    try:
        if isinstance(lookup, dict):
            idx = lookup.get(spectrum_id)
            if idx is None:
                return None
        else:
            idx = lookup.findByNativeID(spectrum_id)
        spec = exp.getSpectrum(idx)
        if spec.getMSLevel() != 2:
            print(f"  ⚠️ Warning: Skipping non-MS2 spectrum in {Path(mzml_file).name} (Level {spec.getMSLevel()})")
            return None
        return spec
    except Exception:
        return None


def annotate_spectrum_with_theoretical(exp_spec, peptidoform, charge=None, tol=0.02, included_ions=None):
    """Annotate experimental spectrum with theoretical fragments.
    
    Uses the EXACT same annotation functions as the app (from spectrum_engine.py).
    
    Args:
        exp_spec: Experimental MSSpectrum
        peptidoform: Peptide sequence (can include modifications)
        charge: Precursor charge (default: 2)
        tol: Fragment mass tolerance in Da
    
    Returns:
        DataFrame with annotated peaks
    """
    if exp_spec is None or exp_spec.size() == 0:
        return pd.DataFrame({"mz": [], "intensity": [], "annotation": []})
    
    try:
        pf = Peptidoform(str(peptidoform))
    except Exception:
        # Fallback: return unannotated spectrum
        df = convert_spectrum_to_df(exp_spec)
        if "annotation" not in df.columns:
            df["annotation"] = "unassigned"
        return df[["mz", "intensity", "annotation"]]
    
    # Safe charge handling
    try:
        if charge is None or pd.isna(charge):
            max_charge = 2
        else:
            max_charge = int(charge)
    except Exception:
        max_charge = 2
    
    # Use app's annotation functions
    theo_spec = generate_theoretical_spectrum(pf, max_charge=max_charge, included_ions=included_ions)
    theo_df = convert_spectrum_to_df(theo_spec)
    pairs = align_spectra(exp_spec, theo_spec, tol)
    aligned_df = annotate_peaks(exp_spec, theo_df, pairs)
    
    return extract_annotations(exp_spec, aligned_df)


print("✅ Helper functions loaded")
print("   Note: All core logic uses spectrum_analysis.py functions")


✅ Helper functions loaded
   Note: All core logic uses spectrum_analysis.py functions


## Step 1: Discover Files

**App equivalent**: File browser UI / file upload

Recursively scan input folders for mzML and idXML files.

**Note**: This step is notebook-specific. In the app, users browse/upload files through the UI.


In [6]:
print("🔍 Discovering files in input folders...")

mzml_paths, idxml_paths = collect_mzml_and_idxml(input_folders)

# Display files
if mzml_paths:
    print("\nmzML files:")
    for p in mzml_paths[:10]:
        print(f"  - {Path(p).name}")
    if len(mzml_paths) > 10:
        print(f"  ... and {len(mzml_paths) - 10} more")

if idxml_paths:
    print("\nidXML files:")
    for p in idxml_paths[:10]:
        print(f"  - {Path(p).name}")
    if len(idxml_paths) > 10:
        print(f"  ... and {len(idxml_paths) - 10} more")

# Resolve reference mzML files
reference_mzml_paths = []
for ref_mzml in reference_mzml_files:
    ref_path = Path(ref_mzml)
    if ref_path.exists():
        reference_mzml_paths.append(str(ref_path))
    else:
        # Try to match by filename
        matches = [p for p in mzml_paths if Path(p).name == ref_mzml]
        if matches:
            reference_mzml_paths.append(matches[0])
        else:
            print(f"  ⚠️ Warning: Reference mzML not found: {ref_mzml}")

reference_mzml_names = [Path(p).name for p in reference_mzml_paths]

print(f"\n📋 Reference mzML files ({len(reference_mzml_names)}):")
for name in reference_mzml_names:
    print(f"  - {name}")


🔍 Discovering files in input folders...

mzML files:
  - 231204_DF_ProCa_ProT001_W6-32_20_LC005-2_MSV035_HT_Rep1_Slot1-19_1_320.mzML
  - 231204_DF_ProCa_ProT001_W6-32_20_LC005-2_MSV035_HT_Rep2_Slot1-19_1_321.mzML
  - 231204_DF_ProCa_ProT001_W6-32_20_LC005-2_MSV035_HT_Rep3_Slot1-19_1_322.mzML
  - 231204_DF_ProCa_ProT002_W6-32_20_LC005-2_MSV035_HT_Rep1_Slot1-21_1_328.mzML
  - 231204_DF_ProCa_ProT002_W6-32_20_LC005-2_MSV035_HT_Rep2_Slot1-21_1_329.mzML
  - 231204_DF_ProCa_ProT002_W6-32_20_LC005-2_MSV035_HT_Rep3_Slot1-21_1_330.mzML
  - 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML

idXML files:
  - ProC_synpeps_a_pout_filtered.idXML
  - ProT001_1_pout_filtered.idXML
  - ProT002_1_pout_filtered.idXML

📋 Reference mzML files (1):
  - 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML


## Step 2: Filter idXMLs by Peptide List

In [7]:
print("🔬 Filtering idXML files by peptide list...")
print(f"   Target peptides: {len(peptides)}")

idxml_to_mzml_spectra, filter_messages, filter_empty = filter_idxml_by_selected_peptides(
    idxml_paths,
    mzml_paths,
    peptides,
    progress_callback=None,
)

if filter_empty:
    print(f"   ⚠️ {len(filter_empty)} files were empty")

# Show mappings
if filter_messages:
    print("\nMappings:")
    for msg in filter_messages[:10]:
        print(f"  {msg}")

🔬 Filtering idXML files by peptide list...
   Target peptides: 4
[DEBUG] Processing ProC_synpeps_a_pout_filtered.idXML: Found 1 run identifiers
[DEBUG]   Run ID: Percolator_2025-12-08T18:13:37_15288616797587562282 -> 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML
[DEBUG]   Priority 2 (Exact Run ID): Percolator_2025-12-08T18:13:37_15288616797587562282 -> 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML
[DEBUG] Match: ATYSVGM(Oxidation)QK (ID: Percolator_2025-12-08T18:13:37_15288616797587562282) -> 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML
[DEBUG]   Priority 2 (Exact Run ID): Percolator_2025-12-08T18:13:37_15288616797587562282 -> 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML
[DEBUG] Match: ATYSVGMQK (ID: Percolator_2025-12-08T18:13:37_15288616797587562282) -> 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML
[DEBUG

### Debug: Peptide Coverage Check

Compare filtered results vs. manual idXML scan to verify correct peptide matching.

In [8]:
print("🔍 Building peptide occurrence table from filtered results...")

# === Use app's efficient filtered results instead of re-scanning idXML files ===
# The filter_idxml_by_selected_peptides() already did the work efficiently
all_occurrences = []

for idxml_name, mzml_map in idxml_to_mzml_spectra.items():
    for mzml_file, spec_list in mzml_map.items():
        for item in spec_list:
            spec_id = item[0]
            peptidoform = item[1]
            # Extract unmodified sequence
            try:
                pf = Peptidoform(peptidoform)
                unmodified = pf.sequence
                
                # Extract modifications dynamically
                modifications = []
                for mod in pf.parsed_sequence:
                    if mod[1]:  # mod[1] contains the modification if present
                        mod_info = str(mod[1])
                        # Extract position and modification name
                        modifications.append(f"{mod[0]}{mod_info}")
                
                modifications_str = "; ".join(modifications) if modifications else "None"
                
            except Exception as e:
                unmodified = peptidoform
                modifications_str = "Modified"
            
            all_occurrences.append({
                'target_peptide': unmodified,
                'peptidoform': peptidoform,
                'modifications': modifications_str,
                'idxml_file': idxml_name,
                'mzml_file': mzml_file,
                'spectrum_id': spec_id,
            })

# Create DataFrame
peptide_occurrence_df = pd.DataFrame(all_occurrences)

# Reorder columns for better readability
column_order = ['target_peptide', 'peptidoform', 'modifications', 'idxml_file', 'mzml_file', 'spectrum_id']
peptide_occurrence_df = peptide_occurrence_df[column_order]

# Save to Excel
output_file = analysis_output_dir / "peptide_occurrences.xlsx"
peptide_occurrence_df.to_excel(output_file, index=False)

# Display summary with better breakdown
print(f"✅ Found {len(peptide_occurrence_df)} total spectrum identifications")

print(f"   Total identifications: {len(peptide_occurrence_df)}")
print(f"   Modified identifications: {(peptide_occurrence_df['modifications'] != 'None').sum()}")
print(f"   Unmodified identifications: {(peptide_occurrence_df['modifications'] == 'None').sum()}")


print(f"\n💾 Saved to: {output_file}")

🔍 Building peptide occurrence table from filtered results...
✅ Found 7 total spectrum identifications
   Total identifications: 7
   Modified identifications: 1
   Unmodified identifications: 6

💾 Saved to: /mnt/volume/nacho/specviz/spectrum_visualizer/analysis/Anna_ProCa_classI_mini2/peptide_occurrences.xlsx


In [9]:
print("🔍 Building peptidoform presence/absence matrix...")

# Create a matrix showing which peptidoforms are in which idXML files
peptidoform_matrix = peptide_occurrence_df.groupby(['peptidoform', 'idxml_file']).size().unstack(fill_value=0)

# Convert counts to checkmarks/crosses
peptidoform_matrix = peptidoform_matrix.applymap(lambda x: '✓' if x > 0 else '✗')

# Add a summary column showing total files with this peptidoform
peptidoform_matrix['Total_Files'] = (peptidoform_matrix == '✓').sum(axis=1)

# Add rows for completely missing target peptides (not found in any idXML)
found_peptides = set(peptide_occurrence_df['target_peptide'].unique())
missing_peptides = [pep for pep in peptides if pep not in found_peptides]

if missing_peptides:
    # Get list of idXML columns (excluding Total_Files)
    idxml_columns = [col for col in peptidoform_matrix.columns if col != 'Total_Files']
    
    # Create rows for missing peptides
    for missing_pep in missing_peptides:
        # Create a row with crosses for all idXML files
        missing_row = {col: '✗' for col in idxml_columns}
        # Use the peptide sequence as the peptidoform (since we don't have modification info)
        peptidoform_matrix.loc[missing_pep] = missing_row

# Sort by total files (descending) and then by peptidoform name
peptidoform_matrix = peptidoform_matrix.sort_values(['Total_Files', peptidoform_matrix.index.name], ascending=[False, True])

# Add a "Total peptides" row at the bottom showing count per idXML
total_row = {}
for col in peptidoform_matrix.columns:
    if col == 'Total_Files':
        total_row[col] = len(peptidoform_matrix)
    else:
        total_row[col] = (peptidoform_matrix[col] == '✓').sum()

peptidoform_matrix.loc['TOTAL_PEPTIDES'] = total_row

# Save to Excel
matrix_output_file = analysis_output_dir / "peptidoform_presence_matrix.xlsx"
peptidoform_matrix.to_excel(matrix_output_file)

print(f"   Peptidoforms found: {len(peptidoform_matrix) - len(missing_peptides) - 1}")  # -1 for TOTAL_PEPTIDES row
print(f"   Total peptidoforms: {len(peptidoform_matrix)-1}")

spectrum_table = build_spectrum_table_from_filtered(idxml_to_mzml_spectra)

# Show breakdown by idXML source
print(f"\nBreakdown by idXML source:")
for idxml_name, count in spectrum_table["idxml_source"].value_counts().items():
    print(f"  {idxml_name}: {count} spectra")

# Show breakdown by mzML file
print(f"\nBreakdown by mzML file:")
for mzml_name, count in spectrum_table["mzml_file"].value_counts().items():
    print(f"  {mzml_name}: {count} spectra")

print(f"\n💾 Saved to: {matrix_output_file}")


🔍 Building peptidoform presence/absence matrix...
   Peptidoforms found: 5
   Total peptidoforms: 5

Breakdown by idXML source:
  ProC_synpeps_a_pout_filtered.idXML: 5 spectra
  ProT001_1_pout_filtered.idXML: 1 spectra
  ProT002_1_pout_filtered.idXML: 1 spectra

Breakdown by mzML file:
  251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML: 5 spectra
  231204_DF_ProCa_ProT001_W6-32_20_LC005-2_MSV035_HT_Rep1_Slot1-19_1_320.mzML: 1 spectra
  231204_DF_ProCa_ProT002_W6-32_20_LC005-2_MSV035_HT_Rep2_Slot1-21_1_329.mzML: 1 spectra

💾 Saved to: /mnt/volume/nacho/specviz/spectrum_visualizer/analysis/Anna_ProCa_classI_mini2/peptidoform_presence_matrix.xlsx


## Step 4: Load Required Spectra

Load only the spectra specified in the spectrum table - uses OnDiscMSExperiment with parallel processing.


In [10]:
print("📂 Loading required spectra from mzML files...")

# Determine which mzML files we need
needed_mzmls = sorted(set(spectrum_table["mzml_file"].tolist()))
print(f"   Needed mzML files from spectrum_table: {needed_mzmls}")

mzml_paths_filtered = [p for p in mzml_paths if Path(p).name in needed_mzmls]
print(f"   Matched {len(mzml_paths_filtered)} mzML file paths (out of {len(mzml_paths)} total)")

# DEBUG: Show which files were matched
if mzml_paths_filtered:
    print(f"   Matched files:")
    for p in mzml_paths_filtered:
        print(f"     - {Path(p).name}")

# Check for unmatched
unmatched = set(needed_mzmls) - set(Path(p).name for p in mzml_paths_filtered)
if unmatched:
    print(f"   ⚠️ WARNING: Could not find paths for {len(unmatched)} mzML files:")
    for name in unmatched:
        print(f"     - {name}")

# Load spectra
mapping_df, validation_info, load_messages, load_empty = load_only_needed_spectra(
    idxml_paths,
    mzml_paths_filtered,
    idxml_to_mzml_spectra,
    progress_callback=None,
)

print(f"\n✅ Loading complete")
print(f"   Loaded {len(mapping_df)} spectra")

# Save mapping table to Excel
if not mapping_df.empty:
    mapping_output_file = analysis_output_dir / "spectrum_mapping.xlsx"
    mapping_df.to_excel(mapping_output_file, index=False)
    print(f"   💾 Saved spectrum mapping to: {mapping_output_file}")

# Check breakdown by idXML source
if not mapping_df.empty:
    print("\nBreakdown by idXML source:")
    for idxml_name, count in mapping_df["idxml_source"].value_counts().items():
        print(f"  {idxml_name}: {count} spectra")

# Show messages if any
if load_messages:
    print("\nMessages (last 20):")
    for msg in load_messages[-20:]:
        print(f"  {msg}")

📂 Loading required spectra from mzML files...
   Needed mzML files from spectrum_table: ['231204_DF_ProCa_ProT001_W6-32_20_LC005-2_MSV035_HT_Rep1_Slot1-19_1_320.mzML', '231204_DF_ProCa_ProT002_W6-32_20_LC005-2_MSV035_HT_Rep2_Slot1-21_1_329.mzML', '251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML']
   Matched 3 mzML file paths (out of 7 total)
   Matched files:
     - 231204_DF_ProCa_ProT001_W6-32_20_LC005-2_MSV035_HT_Rep1_Slot1-19_1_320.mzML
     - 231204_DF_ProCa_ProT002_W6-32_20_LC005-2_MSV035_HT_Rep2_Slot1-21_1_329.mzML
     - 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML

✅ Loading complete
   Loaded 7 spectra
   💾 Saved spectrum mapping to: /mnt/volume/nacho/specviz/spectrum_visualizer/analysis/Anna_ProCa_classI_mini2/spectrum_mapping.xlsx

Breakdown by idXML source:
  ProC_synpeps_a_pout_filtered.idXML: 5 spectra
  ProT002_1_pout_filtered.idXML: 1 spectra
  ProT001_1_pout_filtered.idXML: 1 spectra

Messages (las

## Step 5: Load mzML Files for Plotting

Open mzML files with indexed access and build spectrum lookups:

In [11]:
print("🗄️ Opening mzML files with indexed access...")

experiments = load_indexed_experiments_for_plotting(mzml_paths_filtered)
lookups = build_spectrum_lookups(experiments)

print(f"\n✅ Built {len(lookups)} spectrum lookups")


🗄️ Opening mzML files with indexed access...
  ✅ Opened 231204_DF_ProCa_ProT001_W6-32_20_LC005-2_MSV035_HT_Rep1_Slot1-19_1_320.mzML (109844 spectra)
  ✅ Opened 231204_DF_ProCa_ProT002_W6-32_20_LC005-2_MSV035_HT_Rep2_Slot1-21_1_329.mzML (95392 spectra)
  ✅ Opened 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML (169100 spectra)

✅ Built 3 spectrum lookups


## Step 6: Build Reference Lookup

**App equivalent**: "One-vs-All Replicates" mode

Create a lookup table for reference spectra from selected mzML files (reference runs).
The app allows selecting reference RUNS, then compares all other runs against them for the same peptides.


In [12]:
print(f"📚 Building reference lookup from {len(reference_mzml_names)} reference mzML file(s)...")

# Filter to reference mzML files only
reference_rows = mapping_df[mapping_df["mzml_file"].isin(reference_mzml_names)].copy()

print(f"   Found {len(reference_rows)} reference spectra")

# CRITICAL FIX: Build lookup by UNMODIFIED peptide sequence
# The mapping_df has "peptide_sequence" which contains the FULL peptidoform (with modifications)
# We need to extract the unmodified sequence for matching
reference_by_unmodified = {}
for _, row in reference_rows.iterrows():
    # Extract unmodified sequence from the peptidoform
    try:
        pf = Peptidoform(row["peptide_sequence"])
        unmodified_seq = pf.sequence
    except Exception:
        # Fallback: use the peptidoform as-is
        unmodified_seq = row["peptide_sequence"]
    
    reference_by_unmodified.setdefault(unmodified_seq, []).append(row)

print(f"   Covering {len(reference_by_unmodified)} unique unmodified peptides")


def pick_reference_row(sample_row):
    """Pick best reference row for a given sample row.
    
    Args:
        sample_row: Row from mapping_df representing a sample spectrum
    
    Returns:
        Best matching reference row or None
    """
    # Extract unmodified sequence from sample peptidoform
    try:
        pf = Peptidoform(sample_row["peptide_sequence"])
        unmodified_seq = pf.sequence
    except Exception:
        unmodified_seq = sample_row["peptide_sequence"]
    
    candidates = reference_by_unmodified.get(unmodified_seq, [])
    if not candidates:
        return None
    
    # If charge specified, try to match
    sample_charge = sample_row.get("charge")
    if sample_charge is not None and not pd.isna(sample_charge):
        for row in candidates:
            if "charge" in row and row["charge"] == sample_charge:
                return row
    
    # Return first candidate
    return candidates[0]


print("\n✅ Reference lookup ready")


📚 Building reference lookup from 1 reference mzML file(s)...
   Found 5 reference spectra
   Covering 5 unique unmodified peptides

✅ Reference lookup ready


In [13]:
# DEBUG: Check what's in the mapping_df
print("🔍 DEBUG: Checking mapping_df structure...")
print(f"\nColumns: {list(mapping_df.columns)}")
print(f"\nFirst few rows:")
display(mapping_df.head(3))
print(f"\nSample peptide_sequence values:")
for seq in mapping_df["peptide_sequence"].head(5):
    print(f"  - {seq}")
    try:
        pf = Peptidoform(seq)
        print(f"    → Unmodified: {pf.sequence}")
    except:
        print(f"    → Could not parse")
print(f"\nReference idXML rows:")
print(f"  Total in mapping_df: {len(mapping_df)}")
# print(f"  Reference rows: {len(mapping_df[mapping_df['idxml_source'] == reference_idxml_name])}")

🔍 DEBUG: Checking mapping_df structure...

Columns: ['spectrum_index', 'spectrum_id', 'peptide_sequence', 'charge', 'run', 'score', 'mz', 'rt', 'idxml_source', 'mzml_file']

First few rows:


,spectrum_index,spectrum_id,peptide_sequence,charge,run,score,mz,rt,idxml_source,mzml_file
0,0,index=26194,ATYSVGM(Oxidation)QK,1,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...,0.00026,1000.476423,1046.3,ProC_synpeps_a_pout_filtered.idXML,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...
1,1,index=28062,ATYSVGMQK,1,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...,0.00026,984.483115,1075.3,ProC_synpeps_a_pout_filtered.idXML,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...
2,2,index=33591,ATELTHEDY,1,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...,0.00026,1078.465668,1151.9,ProC_synpeps_a_pout_filtered.idXML,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...



Sample peptide_sequence values:
  - ATYSVGM(Oxidation)QK
    → Could not parse
  - ATYSVGMQK
    → Unmodified: ATYSVGMQK
  - ATELTHEDY
    → Unmodified: ATELTHEDY
  - KPFDQTTISL
    → Unmodified: KPFDQTTISL
  - AYIKNGQLF
    → Unmodified: AYIKNGQLF

Reference idXML rows:
  Total in mapping_df: 7


## Step 7: Generate Mirror Plots

Create mirror plots with reference spectrum on bottom:

In [14]:
# DEBUG: Inspect reference rows in detail
print("🔍 DEBUG: Inspecting reference rows...")
print(f"\nReference rows (all columns):")
display(reference_rows[['peptide_sequence', 'spectrum_id', 'mzml_file', 'idxml_source']])

print(f"\n\nReference lookup dictionary keys (unmodified peptide sequences):")
for i, peptide in enumerate(list(reference_by_unmodified.keys())[:10]):
    print(f"  {i+1}. {peptide}")

print(f"\n\nSample peptides (unmodified sequences):")
sample_rows = mapping_df[~mapping_df["mzml_file"].isin(reference_mzml_names)].copy()
sample_peptides = set()
for _, row in sample_rows.iterrows():
    try:
        pf = Peptidoform(row["peptide_sequence"])
        unmod = pf.sequence
    except:
        unmod = row["peptide_sequence"]
    sample_peptides.add(unmod)

for i, pep in enumerate(sorted(sample_peptides)[:15]):
    has_ref = "✅" if pep in reference_by_unmodified else "❌"
    print(f"  {has_ref} {pep}")

if len(sample_peptides) > 15:
    print(f"  ... and {len(sample_peptides) - 15} more")

# Show overlap analysis
ref_peptides = set(reference_by_unmodified.keys())
overlap = ref_peptides & sample_peptides
print(f"\n\n📊 Overlap Analysis:")
print(f"  Reference peptides: {len(ref_peptides)}")
print(f"  Sample peptides: {len(sample_peptides)}")
print(f"  Overlapping peptides: {len(overlap)}")

if overlap:
    print(f"\n  Overlapping peptides:")
    for pep in sorted(overlap):
        print(f"    ✅ {pep}")
else:
    print(f"\n  ⚠️ NO OVERLAP! Reference and samples have completely different peptides.")
    print(f"\n  💡 Solution: Choose reference mzML files that contain the SAME peptides as your samples.")
    print(f"     Available mzML files with data:")
    for mzml in mapping_df["mzml_file"].unique():
        if mzml not in reference_mzml_names:
            count = len(mapping_df[mapping_df["mzml_file"] == mzml])
            print(f"       - {mzml} ({count} spectra)")


🔍 DEBUG: Inspecting reference rows...

Reference rows (all columns):


,peptide_sequence,spectrum_id,mzml_file,idxml_source
0,ATYSVGM(Oxidation)QK,index=26194,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...,ProC_synpeps_a_pout_filtered.idXML
1,ATYSVGMQK,index=28062,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...,ProC_synpeps_a_pout_filtered.idXML
2,ATELTHEDY,index=33591,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...,ProC_synpeps_a_pout_filtered.idXML
3,KPFDQTTISL,index=70956,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...,ProC_synpeps_a_pout_filtered.idXML
4,AYIKNGQLF,index=74040,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...,ProC_synpeps_a_pout_filtered.idXML




Reference lookup dictionary keys (unmodified peptide sequences):
  1. ATYSVGM(Oxidation)QK
  2. ATYSVGMQK
  3. ATELTHEDY
  4. KPFDQTTISL
  5. AYIKNGQLF


Sample peptides (unmodified sequences):
  ✅ ATELTHEDY
  ✅ KPFDQTTISL


📊 Overlap Analysis:
  Reference peptides: 5
  Sample peptides: 2
  Overlapping peptides: 2

  Overlapping peptides:
    ✅ ATELTHEDY
    ✅ KPFDQTTISL


In [15]:
print("🎨 Generating mirror plots...")
print(f"   Max plots: {max_plots if max_plots else 'all'}")
print(f"   Save to disk: {save_plots}")
print(f"   Using create_matplotlib_mirror_plot() from spectrum_engine.py (same as app)")

plot_count = 0
missing_ref = 0
missing_upper = 0
missing_lower = 0

# DEBUG: Track first few attempts
debug_count = 0
max_debug = 5

for _, row in mapping_df.iterrows():
    # Check plot limit
    if max_plots is not None and plot_count >= max_plots:
        break
    
    # Skip if this IS from a reference mzML file (we want to plot sample vs reference, not reference vs itself)
    if row["mzml_file"] in reference_mzml_names:
        continue
    
    # DEBUG: Show details for first few attempts
    if debug_count < max_debug:
        print(f"\n🔍 DEBUG attempt {debug_count + 1}:")
        print(f"   Sample: {row['peptide_sequence']} from {row['idxml_source']}")
        print(f"   mzML: {row['mzml_file']}")
        print(f"   spectrum_id: {row['spectrum_id']}")
    
    # Find reference row using the improved matching function
    ref_row = pick_reference_row(row)
    if ref_row is None:
        if debug_count < max_debug:
            print(f"   ❌ No reference found")
        missing_ref += 1
        debug_count += 1
        continue
    
    if debug_count < max_debug:
        print(f"   ✅ Found reference:")
        print(f"      Ref peptide: {ref_row['peptide_sequence']}")
        print(f"      Ref mzML: {ref_row['mzml_file']}")
        print(f"      Ref spectrum_id: {ref_row['spectrum_id']}")
    
    # Load upper spectrum (sample) - uses same loading as app
    upper_spec = load_spectrum_from_experiment(experiments, lookups, row["mzml_file"], row["spectrum_id"])
    if upper_spec is None:
        if debug_count < max_debug:
            print(f"   ❌ Failed to load upper (sample) spectrum")
        missing_upper += 1
        debug_count += 1
        continue
    
    if debug_count < max_debug:
        print(f"   ✅ Loaded upper spectrum: {upper_spec.size()} peaks")
    
    # Load lower spectrum (reference) - uses same loading as app
    lower_spec = load_spectrum_from_experiment(experiments, lookups, ref_row["mzml_file"], ref_row["spectrum_id"])
    if lower_spec is None:
        if debug_count < max_debug:
            print(f"   ❌ Failed to load lower (reference) spectrum")
            print(f"      Checking experiments dict: {ref_row['mzml_file'] in experiments}")
            print(f"      Checking lookups dict: {Path(ref_row['mzml_file']).stem in lookups}")
            if Path(ref_row['mzml_file']).stem in lookups:
                lookup = lookups[Path(ref_row['mzml_file']).stem]
                print(f"      Lookup type: {type(lookup)}")
                if isinstance(lookup, dict):
                    print(f"      spectrum_id in lookup dict: {ref_row['spectrum_id'] in lookup}")
                else:
                    try:
                        idx = lookup.findByNativeID(ref_row['spectrum_id'])
                        print(f"      Lookup found index: {idx}")
                    except Exception as e:
                        print(f"      Lookup error: {e}")
        missing_lower += 1
        debug_count += 1
        continue
    
    if debug_count < max_debug:
        print(f"   ✅ Loaded lower spectrum: {lower_spec.size()} peaks")
    
    debug_count += 1
    
    # Annotate spectra - uses EXACT same functions as app
    # (generate_theoretical_spectrum, align_spectra, annotate_peaks, extract_annotations)
    upper_df = annotate_spectrum_with_theoretical(
        upper_spec,
        row["peptide_sequence"],  # Use full peptidoform from mapping_df
        charge=row.get("charge"),
        tol=fragment_mass_tolerance, included_ions=ion_types
    )
    
    lower_df = annotate_spectrum_with_theoretical(
        lower_spec,
        ref_row["peptide_sequence"],  # Use full peptidoform from mapping_df
        charge=ref_row.get("charge"),
        tol=fragment_mass_tolerance, included_ions=ion_types
    )
    
    # Create plot title
    ref_display = ref_row['run'] if 'run' in ref_row else Path(ref_row['mzml_file']).stem
    title = (
        f"{row['run']} {row['peptide_sequence']} (source: {row['idxml_source']})\n"
        f"Reference: {ref_display}"
    )
    
    # Create mirror plot - uses EXACT same function and colors as app
    # For exp vs exp comparison, both spectra use EXP_COLORS (just like the app does)
    fig = create_matplotlib_mirror_plot(
        upper_df,
        lower_df,
        upper_colors=EXP_COLORS,
        lower_colors=EXP_COLORS,
        title=title,
        upper_label="Sample",
        lower_label="Reference",
    )
    
    if fig is None:
        continue
    
    # Save plot if requested - uses EXACT same export function as app
    if save_plots:
        plot_output_dir.mkdir(parents=True, exist_ok=True)
        # Clean spectrum_id for filename (remove special characters)
        clean_spec_id = row['spectrum_id'].replace('/', '_').replace('\\', '_').replace('=', '_')
        filename_base = f"{row['run']}_{clean_spec_id}"
        export_matplotlib_mirror_plot(fig, str(plot_output_dir), filename_base)
    
    # Display plot
    display(fig)
    plot_count += 1

print(f"\n✅ Rendered {plot_count} mirror plot(s)")
print(f"   Missing reference rows: {missing_ref}")
print(f"   Missing upper spectra: {missing_upper}")
print(f"   Missing lower spectra: {missing_lower}")

if save_plots and plot_count > 0:
    print(f"\n💾 Plots saved to: {plot_output_dir}")

🎨 Generating mirror plots...
   Max plots: all
   Save to disk: True
   Using create_matplotlib_mirror_plot() from spectrum_engine.py (same as app)

🔍 DEBUG attempt 1:
   Sample: KPFDQTTISL from ProT002_1_pout_filtered.idXML
   mzML: 231204_DF_ProCa_ProT002_W6-32_20_LC005-2_MSV035_HT_Rep2_Slot1-21_1_329.mzML
   spectrum_id: index=35947
   ✅ Found reference:
      Ref peptide: KPFDQTTISL
      Ref mzML: 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML
      Ref spectrum_id: index=70956
   ✅ Loaded upper spectrum: 391 peaks
   ✅ Loaded lower spectrum: 2489 peaks


<Figure size 3600x2400 with 0 Axes>


🔍 DEBUG attempt 2:
   Sample: ATELTHEDY from ProT001_1_pout_filtered.idXML
   mzML: 231204_DF_ProCa_ProT001_W6-32_20_LC005-2_MSV035_HT_Rep1_Slot1-19_1_320.mzML
   spectrum_id: index=19450
   ✅ Found reference:
      Ref peptide: ATELTHEDY
      Ref mzML: 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML
      Ref spectrum_id: index=33591
   ✅ Loaded upper spectrum: 305 peaks
   ✅ Loaded lower spectrum: 220 peaks


<Figure size 3600x2400 with 0 Axes>


✅ Rendered 2 mirror plot(s)
   Missing reference rows: 0
   Missing upper spectra: 0
   Missing lower spectra: 0

💾 Plots saved to: /mnt/volume/nacho/specviz/spectrum_visualizer/plots/Anna_ProCa_classI_mini2


## Step 8: Calculate One-vs-All Summary

Calculate R-squared values for all samples against their best reference match and generate summary statistics and plots.

In [16]:
print("📊 Calculating One-vs-All statistics...")

# Initialize results container
r2_results = []
summary_plot_output_dir = specviz_dir / "plots" / project_name / "summary"
summary_plot_output_dir.mkdir(parents=True, exist_ok=True)

# Loop through all filtered peptides
for peptide_seq in peptides:
    # Get all rows for this peptide
    peptide_rows = mapping_df[mapping_df["peptide_sequence"].str.contains(peptide_seq, regex=False)]
    
    if peptide_rows.empty:
        continue
        
    # Separate into reference and samples
    ref_rows = peptide_rows[peptide_rows["mzml_file"].isin(reference_mzml_names)]
    sample_rows = peptide_rows[~peptide_rows["mzml_file"].isin(reference_mzml_names)]
    
    if ref_rows.empty or sample_rows.empty:
        continue
        
    # For summary, we compare EACH sample against the BEST reference
    for _, sample_row in sample_rows.iterrows():
        ref_row = pick_reference_row(sample_row)
        if ref_row is None:
            continue
            
        # Load spectra
        sample_spec = load_spectrum_from_experiment(experiments, lookups, sample_row["mzml_file"], sample_row["spectrum_id"])
        ref_spec = load_spectrum_from_experiment(experiments, lookups, ref_row["mzml_file"], ref_row["spectrum_id"])
        
        if sample_spec is None or ref_spec is None:
            continue
            
        # Annotate
        # Note: We assume spectra loaded from experiments are raw and need annotation
        sample_df = annotate_spectrum_with_theoretical(
             sample_spec, sample_row["peptide_sequence"], charge=sample_row.get("charge"), 
             tol=fragment_mass_tolerance, included_ions=ion_types
        )
        ref_df = annotate_spectrum_with_theoretical(
             ref_spec, ref_row["peptide_sequence"], charge=ref_row.get("charge"), 
             tol=fragment_mass_tolerance, included_ions=ion_types
        )
        
        # Calculate R2
        # Method 1: Matching
        sample_ions = [(r.mz, r.intensity, r.annotation) for _, r in sample_df.iterrows() if r.annotation != 'unassigned']
        ref_ions = [(r.mz, r.intensity, r.annotation) for _, r in ref_df.iterrows() if r.annotation != 'unassigned']
        
        r2_matching = None
        if sample_ions and ref_ions:
             r2_matching, _, _, _ = spectrum_correlation(tuple(sample_ions), tuple(ref_ions))
             
        # Method 2: Binned
        r2_binned = None
        if not sample_df.empty and not ref_df.empty:
             r2_binned, _, _, _ = new_spectrum_correlation(sample_df, ref_df)
             
        # Store result
        r2_results.append({
            "peptidoform": peptide_seq,
            "sample_run": sample_row['run'],
            "reference_run": ref_row['run'],
            "r2_matching": r2_matching if r2_matching is not None else 0,
            "r2_binned": r2_binned if r2_binned is not None else 0
        })

r2_data = pd.DataFrame(r2_results)
print(f"✅ Calculated statistics for {len(r2_data)} comparisons")


📊 Calculating One-vs-All statistics...
✅ Calculated statistics for 2 comparisons


In [17]:
if not r2_data.empty:
    print("📈 Generating overview summary plot...")
    
    # Create overview plot
    # Note: create_peptide_overview_plot expects 'r2_matching' or 'r2_binned' column
    fig, summary_df = create_peptide_overview_plot(
        r2_data=r2_data,
        r2_metric='r2_matching', # Default to matching
        top_n=None
    )
    
    if fig:
        # Save plot
        plot_path = export_overview_plot(fig, str(summary_plot_output_dir), "matching")
        display(fig)
        print(f"💾 Summary plot saved to: {plot_path}")
    else:
        print("⚠️ Failed to create summary plot")
        
    # Export Excel
    excel_path = analysis_output_dir / "summary_r2_data.xlsx"
    r2_data.to_excel(excel_path, index=False)
    print(f"💾 Summary data saved to: {excel_path}")
    
    # Display top 5 rows
    display(r2_data.head())
else:
    print("⚠️ No R2 data to plot")


📈 Generating overview summary plot...


<Figure size 1000x600 with 2 Axes>

💾 Summary plot saved to: /mnt/volume/nacho/specviz/spectrum_visualizer/plots/Anna_ProCa_classI_mini2/summary/peptide_count_r2_matching.png
💾 Summary data saved to: /mnt/volume/nacho/specviz/spectrum_visualizer/analysis/Anna_ProCa_classI_mini2/summary_r2_data.xlsx


,peptidoform,sample_run,reference_run,r2_matching,r2_binned
0,KPFDQTTISL,231204_DF_ProCa_ProT002_W6-32_20_LC005-2_MSV03...,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...,0.000000,0.001648
1,ATELTHEDY,231204_DF_ProCa_ProT001_W6-32_20_LC005-2_MSV03...,251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV...,0.910352,0.139215


## Summary

Pipeline complete! Review the statistics below:

In [18]:
print("="*60)
print("PIPELINE SUMMARY")
print("="*60)

print(f"\n📁 Input:")
print(f"   Folders scanned: {len(input_folders)}")
print(f"   mzML files found: {len(mzml_paths)}")
print(f"   idXML files found: {len(idxml_paths)}")
print(f"   Target peptides: {len(peptides)}")

print(f"\n🔬 Filtering:")
print(f"   idXML files with matches: {len(idxml_to_mzml_spectra)}")
print(f"   Total spectra in table: {len(spectrum_table)}")

print(f"\n📂 Loading:")
print(f"   mzML files loaded: {len(mzml_paths_filtered)}")
print(f"   Spectra mapped: {len(mapping_df)}")

print(f"\n📋 Reference:")
print(f"   Reference mzML files: {len(reference_mzml_names)}")
for name in reference_mzml_names:
    count = len(mapping_df[mapping_df['mzml_file'] == name])
    print(f"     - {name}: {count} spectra")
print(f"   Total reference spectra: {len(reference_rows)}")
print(f"   Reference peptides: {len(reference_by_unmodified)}")
print(f"   Non-reference spectra: {len(mapping_df[~mapping_df['mzml_file'].isin(reference_mzml_names)])}")

print(f"\n🎨 Plotting:")
print(f"   Plots generated: {plot_count}")
print(f"   Missing references: {missing_ref}")
print(f"   Missing sample spectra: {missing_upper}")
print(f"   Missing reference spectra: {missing_lower}")

if save_plots and plot_count > 0:
    print(f"\n💾 Output: {plot_output_dir}")

print("\n" + "="*60)
print("✅ Pipeline complete!")
print("="*60)


PIPELINE SUMMARY

📁 Input:
   Folders scanned: 1
   mzML files found: 7
   idXML files found: 3
   Target peptides: 4

🔬 Filtering:
   idXML files with matches: 3
   Total spectra in table: 7

📂 Loading:
   mzML files loaded: 3
   Spectra mapped: 7

📋 Reference:
   Reference mzML files: 1
     - 251201_AD_ProC_ImmProCure_Synpep_100fmoluL_LCV005-2_MSV035_HT_Slot1-1_1_9333.mzML: 5 spectra
   Total reference spectra: 5
   Reference peptides: 5
   Non-reference spectra: 2

🎨 Plotting:
   Plots generated: 2
   Missing references: 0
   Missing sample spectra: 0
   Missing reference spectra: 0

💾 Output: /mnt/volume/nacho/specviz/spectrum_visualizer/plots/Anna_ProCa_classI_mini2

✅ Pipeline complete!
